# E-commerce Sales - Data Exploration

**Objective:** Explore and understand the raw dataset before cleaning and analysis.

**Dataset:** UCI Online Retail II  
**Author:** Hamdi Ahmed  
**Date:** 27/06/2026

## 1. Setup

In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

# Load both sheets and combine into 1 dataset
df1 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/raw/online_retail_II.xlsx', sheet_name='Year 2010-2011')
raw = pd.concat([df1, df2], ignore_index=True)


con = duckdb.connect()
con.register('raw_retail', raw)
print(f"Loaded {len(raw):,} rows into DuckDB")


Loaded 1,067,371 rows into DuckDB


In [3]:
# Full Dataset
con.execute("""
    SELECT *
    FROM raw_retail 
    LIMIT 10
""").fetchdf()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


## 2. Basic Structure

In [4]:
# Row count
con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM raw_retail
""").fetchdf()

,total_rows
0,1067371


In [5]:
total_rows = con.execute("""
    SELECT COUNT(*) AS total_rows
    FROM raw_retail
""").fetchdf()['total_rows'][0]

In [6]:
# Column names and types
con.execute("DESCRIBE raw_retail").fetchdf()

,column_name,column_type,null,key,default,extra
0,Invoice,VARCHAR,YES,None,None,None
1,StockCode,VARCHAR,YES,None,None,None
2,Description,VARCHAR,YES,None,None,None
3,Quantity,BIGINT,YES,None,None,None
4,InvoiceDate,TIMESTAMP,YES,None,None,None
5,Price,DOUBLE,YES,None,None,None
6,Customer ID,DOUBLE,YES,None,None,None
7,Country,VARCHAR,YES,None,None,None


In [7]:
# Count of distinct values in each column
con.execute("""
    SELECT
        COUNT(DISTINCT Invoice) AS unique_invoices,
        COUNT(DISTINCT StockCode) AS unique_stockcodes,
        COUNT(DISTINCT Description) AS unique_descriptions,
        COUNT(DISTINCT "Customer ID")  AS unique_customers,
        COUNT(DISTINCT Country) AS unique_countries,
        COUNT(*) AS total_rows
    FROM raw_retail;  
""").fetchdf()

,unique_invoices,unique_stockcodes,unique_descriptions,unique_customers,unique_countries,total_rows
0,53628,5305,5698,5942,43,1067371


In [8]:
# Memory usage
print(f'\n💾 Memory Usage: {raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB')


💾 Memory Usage: 216.61 MB


## 3. Missing & Duplicate Values

In [9]:
# Number of null values in the invoice column
con.execute("""
    SELECT 
        'invoice' as column_name,
        SUM(CASE WHEN invoice IS NULL THEN 1 ELSE 0 END) AS "Number of Nulls",
        ROUND(SUM(CASE WHEN "invoice" IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS "Pct_Nulls"
    FROM 
        raw_retail    
""").fetchdf()

,column_name,Number of Nulls,Pct_Nulls
0,invoice,0.0,0.0


In [10]:
# Automating sql query above to handle multiple columns

df = con.execute("DESCRIBE raw_retail").fetchdf()

column_names = df['column_name'].tolist()

query_parts = []

for column_name in column_names:
    piece = f"""
    SELECT 
        '{column_name}' as column_name,
        SUM(CASE WHEN "{column_name}" IS NULL THEN 1 ELSE 0 END) AS "Number of Nulls",
        ROUND(SUM(CASE WHEN "{column_name}"  IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS "Pct_Nulls"
    FROM 
        raw_retail    
    """
    query_parts.append(piece)
missing_values_query = " UNION ALL ".join(query_parts)

In [11]:
missing_values_df = con.execute(missing_values_query).fetchdf()
missing_values_df

,column_name,Number of Nulls,Pct_Nulls
0,Invoice,0.0,0.00
1,StockCode,0.0,0.00
2,Description,4382.0,0.41
3,Quantity,0.0,0.00
4,InvoiceDate,0.0,0.00
5,Price,0.0,0.00
6,Customer ID,243007.0,22.77
7,Country,0.0,0.00


In [12]:
customer_count = con.execute("""
    SELECT 
        count(DISTINCT "customer ID") as customer_count
    FROM 
        raw_retail
""").fetchdf()['customer_count'][0]

In [13]:
missing_customer_count = con.execute("""
    SELECT 
        count(*) as count
    FROM 
        raw_retail  
    WHERE 
        "Customer ID" IS NULL
""").fetchdf()['count'][0]

#### INSIGHT: 
#### • Only the description and customer ID columns have missing values. 0.41% of description have nulls but nearly a quarter of customer ID values are null. This could be transactions from those not registered as customers such as guests.

#### • Rows with null customer ID contribute to revenue but cannot be used in customer segmentation analysis

### Duplicate Rows

In [14]:
# Concatenating all columns into a single string and counting distinct combinations.
# Gapp between total rows and distinct combinations is the duplicate count.
con.execute("""
    WITH duplicate_count AS (
      SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT
            CAST(Invoice AS VARCHAR)      || '|' ||
            CAST(StockCode AS VARCHAR)    || '|' ||
            CAST(InvoiceDate AS VARCHAR)  || '|' ||
            CAST(Quantity AS VARCHAR)     || '|' ||
            CAST(Price AS VARCHAR)        || '|' ||
            COALESCE(Description, 'NULL') || '|' ||
            CAST(COALESCE("Customer ID", -1) AS VARCHAR) || '|' ||
            COALESCE(Country, 'NULL')
        ) AS exact_duplicates
     FROM raw_retail
    )
    SELECT 
        total_rows,
        exact_duplicates,
        (exact_duplicates/total_rows) * 100 as pct_duplicates
    FROM duplicate_count
""").fetchdf()

,total_rows,exact_duplicates,pct_duplicates
0,1067371,34335,3.216782


In [15]:
duplicates = con.execute("""
    WITH duplicate_count AS (
      SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(DISTINCT
            CAST(Invoice AS VARCHAR)      || '|' ||
            CAST(StockCode AS VARCHAR)    || '|' ||
            CAST(InvoiceDate AS VARCHAR)  || '|' ||
            CAST(Quantity AS VARCHAR)     || '|' ||
            CAST(Price AS VARCHAR)        || '|' ||
            COALESCE(Description, 'NULL') || '|' ||
            CAST(COALESCE("Customer ID", -1) AS VARCHAR) || '|' ||
            COALESCE(Country, 'NULL')
        ) AS exact_duplicates
     FROM raw_retail
    )
    SELECT 
        total_rows,
        exact_duplicates,
        (exact_duplicates/total_rows) * 100 as pct_duplicates
    FROM duplicate_count
""").fetchdf()

In [16]:
## Grouping by all columns to determinine if these duplicates are genuine repeat events

con.execute("""
    SELECT 
        *, COUNT(*) AS n
    FROM 
        raw_retail
    GROUP BY 
        *
    HAVING 
        COUNT(*) > 1
    ORDER BY 
        n DESC
    LIMIT 20;
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,n
0,555524,22698,PINK REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,20
1,555524,22697,GREEN REGENCY TEACUP AND SAUCER,1,2011-06-05 11:37:00,2.95,16923.0,United Kingdom,12
2,537224,70007,HI TEC ALPINE HAND WARMER,1,2010-12-05 16:24:00,1.65,13174.0,United Kingdom,10
3,572861,22775,PURPLE DRAWERKNOB ACRYLIC EDWARDIAN,12,2011-10-26 12:46:00,1.25,14102.0,United Kingdom,8
4,537196,21811,CHRISTMAS HANGING HEART WITH BELL,1,2010-12-05 13:55:00,1.25,15426.0,United Kingdom,6
5,572344,M,Manual,48,2011-10-24 10:43:00,1.50,14607.0,United Kingdom,6
6,537781,22898,CHILDRENS APRON APPLES DESIGN,1,2010-12-08 12:46:00,1.95,17341.0,United Kingdom,6
7,534219,35953,FOLKART STAR CHRISTMAS DECORATIONS,1,2010-11-21 16:13:00,1.25,13230.0,United Kingdom,6
8,537144,21882,SKULLS TAPE,1,2010-12-05 13:00:00,0.65,15880.0,United Kingdom,6
9,502660,17021,NAMASTE SWAGAT INCENSE,6,2010-03-25 17:18:00,0.30,13187.0,United Kingdom,6


#### INSIGHT: 
#### • Duplicates are identical across every column including the InvoiceDate timestamp. The same transaction can't have genuinely happened twice so this is likely a data extraction issue

## 4.Distributions

In [17]:
# Stock Code Distribution
con.execute("""
   SELECT
        LENGTH(CAST(StockCode AS VARCHAR)) AS code_length,
        COUNT(*) AS rows,
        COUNT(DISTINCT StockCode) AS distinct_codes,
        ANY_VALUE(StockCode) AS example
    FROM 
        raw_retail
    GROUP BY 
        code_length
    ORDER BY 
        rows DESC
""").fetchdf()

,code_length,rows,distinct_codes,example
0,5,932385,3590,21755
1,6,127590,1649,90181B
2,4,2158,4,POST
3,1,1713,5,M
4,3,1446,1,DOT
5,7,1393,9,15056BL
6,2,283,2,C2
7,12,202,10,BANK CHARGES
8,8,127,30,DCGSSBOY
9,9,74,5,AMAZONFEE


#### INSIGHT: 
#### • Stock Codes are typically 5 or 6 characters long composed of numbers or numbers plus a letter. Anything less than 5 character looks to be not as useful 

In [18]:
# Price & Quantity Distribution
numerical_columns = ['Quantity', 'Price']

numeric_query_parts = []

for column in numerical_columns:
    piece = f"""
    SELECT 
        '{column}' as column_name,
        COUNT(*) as total,
        ROUND(AVG({column}),2) as mean,
        MIN({column}) as min,
        QUANTILE_CONT({column},0.25) as qntile_25,
        QUANTILE_CONT({column},0.5) as median,
        QUANTILE_CONT({column},0.75) as qntile_75,
        MAX({column}) as max,
    FROM 
        raw_retail    
    """
    numeric_query_parts.append(piece)
numeric_columns_query = " UNION ALL ".join(numeric_query_parts)

In [19]:
numeric_columns_df = con.execute(numeric_columns_query).fetchdf()
numeric_columns_df

,column_name,total,mean,min,qntile_25,median,qntile_75,max
0,Quantity,1067371,9.94,-80995.00,1.00,3.0,10.00,80995.0
1,Price,1067371,4.65,-53594.36,1.25,2.1,4.15,38970.0


In [20]:
# Negative quantities
con.execute("""
    SELECT 
        *
    FROM 
        raw_retail  
    WHERE 
        Quantity < 0
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
1,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
2,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
3,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
4,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
22945,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
22946,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
22947,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
22948,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [21]:
negative_quantities = con.execute("""
    SELECT 
        count(*) as count
    FROM 
        raw_retail  
    WHERE 
        Quantity < 0
""").fetchdf()['count'][0]

In [22]:
# Zero price
con.execute("""
    SELECT 
        *
    FROM 
        raw_retail  
    WHERE 
        Price = 0
    LIMIT 20
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
1,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
2,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
3,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
4,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
5,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
6,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
7,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
8,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
9,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom


In [23]:
# Negative price
con.execute("""
    SELECT 
        *
    FROM 
        raw_retail  
    WHERE 
        Price < 0
    LIMIT 20
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
1,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
2,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
3,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
4,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [24]:
zero_negative_price = con.execute("""
    SELECT 
        count(*) as count
    FROM 
        raw_retail  
    WHERE 
        Price <= 0
    LIMIT 20
""").fetchdf()['count'][0]

In [25]:
con.execute("""
    SELECT 
        *
    FROM 
        raw_retail  
    WHERE 
        Invoice LIKE 'C%'
""").fetchdf()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
1,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
2,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
3,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
4,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
...,...,...,...,...,...,...,...,...
19489,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
19490,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
19491,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
19492,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


In [26]:
cancellations = con.execute("""
    SELECT 
        count(*) as count
    FROM 
        raw_retail  
    WHERE 
        Invoice LIKE 'C%'
""").fetchdf()['count'][0]

#### INSIGHT: 

#### • 0 and negative values exist for quantity and price. Negative quantities are likely returns. Negative price are for bad debt but no clear explanation for 0 price. Could be returns, promotional items, samples or data entry errors. 

#### • Observations with negative or zero price are only 0.6% of the total. Invoices starting with 'C' represent 1.8% of the total

### Top 10 Countries

In [27]:
con.execute("""
    SELECT 
        Country,
        Count(*) as Count
    FROM 
        raw_retail
    GROUP BY
        Country
    ORDER BY
        count DESC
    LIMIT 10
""").fetchdf()

,Country,Count
0,United Kingdom,981330
1,EIRE,17866
2,Germany,17624
3,France,14330
4,Netherlands,5140
5,Spain,3811
6,Switzerland,3189
7,Belgium,3123
8,Portugal,2620
9,Australia,1913


In [28]:
country_count = con.execute("""
    SELECT 
        count(DISTINCT Country) as count
    FROM 
        raw_retail
""").fetchdf()['count'][0]

## 5.Date/Time Analysis

In [29]:
# Full date range and distinct days

date_range = con.execute("""
    SELECT
    MIN(InvoiceDate) AS earliest_date,
    MAX(InvoiceDate) AS latest_date,
    COUNT(DISTINCT CAST(InvoiceDate AS DATE)) AS unique_days
    FROM raw_retail;
""").fetchdf()

date_range

,earliest_date,latest_date,unique_days
0,2009-12-01 07:45:00,2011-12-09 12:50:00,604


In [30]:
earliest_date = date_range['earliest_date'][0]
latest_date = date_range['latest_date'][0]
unique_days = date_range['unique_days'][0]

In [31]:
con.execute("""
    SELECT
    DATE_TRUNC('month', InvoiceDate) AS month,
    COUNT(*) AS no_observations,
    COUNT(DISTINCT CAST(InvoiceDate AS DATE)) AS trading_days
    FROM raw_retail
    GROUP BY month
    ORDER BY month ASC;
""").fetchdf()

,month,no_observations,trading_days
0,2009-12-01,45228,21
1,2010-01-01,31555,24
2,2010-02-01,29388,24
3,2010-03-01,41511,27
4,2010-04-01,34057,23
5,2010-05-01,35323,24
6,2010-06-01,39983,26
7,2010-07-01,33383,26
8,2010-08-01,33306,26
9,2010-09-01,42091,26


#### INSIGHT: The dataset covers 604 unique days from 1st december 2009 to 9th december 2011. No months are missing. The months have similar number of trading days in the 20s except for december 2011 that only had 8 trading days but the number of observations from that month was still in line with rest of the months

In [32]:
print('=' * 60)
print('📊 DATA QUALITY SUMMARY')
print('=' * 60)

print(f"""
DATASET OVERVIEW:
• Total rows: {total_rows:,}
• Date range: {earliest_date.date()} to {latest_date.date()}
• Unique days: {unique_days}
• Countries: {country_count}
• Customers: {customer_count}



DATA QUALITY ISSUES:
1. Missing Customer IDs: {missing_customer_count:,} rows ({missing_customer_count/total_rows*100:.1f}% of total)
   → Likely guest checkouts, exclude from customer analysis but keep for any revenue analysis

2. Exact Duplicates Across Rows: {duplicates['exact_duplicates'][0]:,} rows {duplicates['pct_duplicates'][0]:.1f}% of total)
   → Could be data extraction issue, remove from data as a genuine repeat purchases unlikely to occure twice or more on the same second

3. Cancellations: {cancellations:,} rows ({cancellations/total_rows*100:.1f}% of total)
   → Invoices starting with 'C', flag them in the final dataset

4. Negative Quantities: {negative_quantities:,} rows ({negative_quantities/total_rows*100:.1f}% of total)
   → Returns/cancellations, flag them as cancellation in the final dataset

5. Zero/Negative Prices: {zero_negative_price:,} rows ({zero_negative_price/total_rows*100:.1f}% of total)
   → Negative prices are for bad debt, remove from final data
   → No clear explanation for zero price, could be free items or data entry errors, remove from final data

6. Stock Code Distribution: Mostly 5 or 6 characters long although there are 10 distinct character lengths
   → Observe distribution again after all other cleaning steps are complete.
   → Remove stock codes with less than 5 characters.
   → Look more closely at stock codes with more than 6 characters.
       








NEXT STEPS:
→ Proceed to data_cleaning.ipynb to address these issues
""")

📊 DATA QUALITY SUMMARY

DATASET OVERVIEW:
• Total rows: 1,067,371
• Date range: 2009-12-01 to 2011-12-09
• Unique days: 604
• Countries: 43
• Customers: 5942



DATA QUALITY ISSUES:
1. Missing Customer IDs: 243,007 rows (22.8% of total)
   → Likely guest checkouts, exclude from customer analysis but keep for any revenue analysis

2. Exact Duplicates Across Rows: 34,335 rows 3.2% of total)
   → Could be data extraction issue, remove from data as a genuine repeat purchases unlikely to occure twice or more on the same second

3. Cancellations: 19,494 rows (1.8% of total)
   → Invoices starting with 'C', flag them in the final dataset

4. Negative Quantities: 22,950 rows (2.2% of total)
   → Returns/cancellations, remove as they are not valid sales

5. Zero/Negative Prices: 6,207 rows (0.6% of total)
   → Negative prices are for bad debt, remove from final data
   → No clear explanation for zero price, could be free items or data entry errors, remove from final data

6. Stock Code Distribu